# Basin footprint estimation

Derives one fixed footprint polygon per site from multi-year seasonal behaviour, so that
every month only has to answer *what state is the waterhole in*, not *where is it*. The
footprint is also the denominator for the composition fractions that become the time series.

**Why not a threshold on maximum MNDWI.** Emergent sedges and melaleuca routinely cover
standing water at these sites and drag MNDWI far negative. A water-index threshold would
give a sedge-choked basin no footprint at all — exactly the sites that matter most. So a
basin is found by how a pixel *behaves* across years instead:

- **seasonal range** (wet max − dry min) of several indices — the basin swings with the
  season far more than the savanna matrix does;
- **dry-season NDVI anomaly** — a vegetated basin stays green while the matrix browns off.

Both are robust z-scores against the tile's own matrix, so no absolute threshold has to
hold across sites with different soils and cover.

In [ ]:
import sys
from pathlib import Path

# Modules live alongside this notebook. Works whether the kernel starts here or
# at the repository root.
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "cookie-cutting" else Path.cwd() / "cookie-cutting"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import wh_config
import wh_footprint
import wh_inventory
import wh_plots
import wh_temporal

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

cfg = wh_config.load()
print("config", cfg.source_path.name, "hash", cfg.hash)
print("tiles ", cfg.paths["tiles"])

## Parameters

Everything tunable about footprint derivation, in one place. These are deliberately here
rather than in `waterhole_seg_config.yaml` so they are visible and editable while you are
looking at the output. The values are recorded into every saved GeoJSON, so a footprint
always carries the settings that produced it.

In [ ]:
PARAMS = wh_footprint.FootprintParams(
    # Indices whose SEASONAL RANGE (wet max - dry min) feeds the basin score,
    # mapped to their weight. One dict, so adding or removing an index cannot
    # leave a parallel list of weights out of step. Set a weight to 0.0, or drop
    # the key, to take an index out.
    #
    # Measured at site 025 (basin vs matrix separation):
    #   mndwi_seasonal_range  2.85x   <- carries the signal
    #   ndvi_seasonal_range   1.02x   <- savanna swings as hard as the basin does
    # NDVI still matters here, but as the dry-season ANOMALY below, not as a range.
    seasonal_range_weights={
        "mndwi": 1.0,
        "ndti": 1.0,
        # "ndvi": 1.0,
    },

    # Dry-season NDVI anomaly weight. Carries more than the others because it is
    # the signal that finds the vegetated basins MNDWI cannot see.
    dry_ndvi_anomaly_weight=1.5,

    # A pixel joins the basin above this weighted-mean z-score.
    # Lower  -> larger, more inclusive footprints (watch max_basin_fraction).
    # Higher -> only the strongest-swinging core.
    # Sites near the cliff edge are sensitive: site 119 goes from failing to
    # 49 px between 2.00 and 1.75.
    score_threshold=1.75,

    # Reject a central component smaller than this; below ~4 pixels at 10 m
    # there is not enough signal to call it a basin rather than noise.
    min_basin_pixels=4,

    # Morphological closing before component selection, to bridge one-pixel
    # gaps in a basin that the score just misses.
    closing_radius_px=1,

    # Buffer added after the core is chosen, to catch the drawdown margin
    # where the interesting degradation happens.
    buffer_px=3,

    # How far from the tile centre to look for a candidate. The AOI was buffered
    # from a labelled bounding-box centre, so the basin is near but not exactly
    # at the centre.
    seed_search_radius_px=15,

    # A pixel needs this many observed months for its seasonal statistics to
    # mean anything. 24 of 84 is conservative.
    min_valid_months=24,

    # Sanity ceiling: a "basin" covering more than this share of a 1.5 km tile
    # means the threshold is too low for this site. Reported as a failure rather
    # than silently accepted.
    max_basin_fraction=0.25,
)

# Sites to work through in the exploration cells below.
EXPLORE_SITES = ["002", "003", "025", "000", "075", "119"]

PARAMS

## Load the manifest

In [ ]:
manifest = wh_inventory.load_manifest(cfg)
sites = sorted(manifest["site_id"].unique())
print(f"{len(manifest):,} chips, {len(sites)} sites, "
      f"{manifest['year_month'].nunique()} months")
manifest.head(3)

## Waterhole bounding boxes

The AOIs were cut by buffering each labelled box *centre* by 750 m, so a 1.5 km chip often
catches neighbours: **93 of 187 tiles contain more than one waterhole**, up to 5. A
composition fraction over a whole tile therefore mixes several waterholes together, which
confounds exactly the per-site time series this project exists to produce.

The same labelme JSON carries each waterhole's rectangular *extent*. Rasterising that onto
each tile's grid bounds a site to its own basin.

Two things the box mask is deliberately **not** used for:

- **It never filters training pixels.** `surrounding_vegetation` is 50.7% outside the box by
  definition — it *is* the matrix — while only 10 of 29,314 waterhole-class labelled pixels
  fall outside. Filtering on the box would delete the majority class.
- **It never replaces the tile for statistics.** `robust_z` compares a pixel to its tile's
  median and MAD, and the buffered box is a median 10% of the tile — nowhere near enough
  matrix to estimate a baseline from.

What it *is* for: deciding which connected component belongs to this waterhole, bounding the
composition denominator, and keeping the pseudo-labeller off neighbouring waterholes.

In [ ]:
import wh_bbox

BOX_PARAMS = wh_bbox.BoxParams(
    # Added on EVERY side, so a 292 m box becomes 492 m across — headroom for the
    # waterhole being larger in the wet season than when it was drawn.
    # At 100 m the buffered box covers a median 10% of a tile and 7 of 187 boxes
    # overflow the tile. Keep it modest: the mask must leave enough savanna matrix
    # for the z-score baselines and the 9x9 context windows to work.
    buffer_m=100.0,
)

boxes = wh_bbox.extract_boxes(cfg)
wh_bbox.save_boxes(boxes, cfg)

print(f"{len(boxes)} waterhole boxes")
print(f"size: median {boxes['width_m'].median():.0f} x {boxes['height_m'].median():.0f} m, "
      f"range {boxes['width_m'].min():.0f}-{boxes['width_m'].max():.0f} m")
boxes[["site_id", "label", "width_m", "height_m"]].head()

### Check the boxes really are indexed by site

Row order in the JSON is taken to be the site id. That is not an assumption to leave
unchecked — if the JSON were ever re-saved in a different order, every mask would attach to
the wrong waterhole and nothing downstream would notice. The filenames encode each AOI's
centre, so they can be compared directly.

In [ ]:
problems = wh_bbox.verify_against_tiles(boxes, manifest)
if len(problems):
    print(f"!! {len(problems)} site(s) where the box does not match the tile:")
    display(problems)
else:
    print(f"all {len(boxes)} boxes match their tile's centre — row order is the site id")

In [ ]:
box_summary = wh_bbox.build_all(manifest, cfg, boxes, BOX_PARAMS)
box_summary.to_csv(cfg.paths["derived"] / "bounding_box_summary.csv", index=False)

print("\nfraction of the tile covered by the buffered box:")
print(box_summary["tile_fraction"].describe(percentiles=[.1, .5, .9]).round(3).to_string())

clipped = box_summary[box_summary["clipped_by_tile"]]
if len(clipped):
    print(f"\n{len(clipped)} box(es) overflow the tile and are clipped to it: "
          f"{sorted(clipped['site_id'])}")
    print("For these the mask cannot bound the waterhole, so their composition")
    print("fractions are over the whole tile and mean something different.")
box_summary.head()

### Neighbouring waterholes

How much of each tile belongs to a *different* waterhole. This is the quantity the box mask
exists to exclude.

In [ ]:
import wh_tiles

sample = boxes["site_id"].tolist()[::20]
rows = []
for site_id in sample:
    tile_rows = manifest[manifest["site_id"] == site_id]
    if tile_rows.empty:
        continue
    tile = wh_tiles.read_tile(tile_rows.iloc[0]["tif_path"], cfg)
    own, _ = wh_bbox.box_mask(boxes.loc[site_id], tile, BOX_PARAMS)
    others = wh_bbox.neighbour_mask(site_id, tile, boxes, BOX_PARAMS)
    rows.append({
        "site_id": site_id,
        "own_pct": round(100 * own.mean(), 1),
        "neighbours_pct": round(100 * others.mean(), 1),
    })

pd.DataFrame(rows).set_index("site_id")

## One site, end to end

Loads the site's whole time series, computes the temporal features, and derives the
footprint. Everything below reuses these objects, so re-run this cell when you change
`SITE`.

In [ ]:
SITE = EXPLORE_SITES[0]

footprint, stack, features = wh_footprint.run_site(manifest, SITE, cfg, PARAMS)

print(f"site {SITE}")
print(f"  months loaded : {stack.n_months}  shape {stack.shape}")
print(f"  features      : {len(features)}")
print(f"  footprint     : {footprint.n_pixels} px, {footprint.area_m2/1e4:.2f} ha, "
      f"{100*footprint.fraction_of_tile:.1f}% of tile")
print(f"  status        : {'OK' if footprint.succeeded else 'FAILED — ' + footprint.reason}")
if footprint.notes:
    print(f"  note          : {footprint.notes}")

## Diagnostics: what produced the footprint

Top row is what your eye can check — wet and dry true colour with the derived outline
(solid magenta = buffered footprint, dashed white = unbuffered core). Bottom row is what
the algorithm actually used: each anomaly layer as a z-score, and the count of observed
months.

If the outline looks wrong, the bottom row tells you which layer to blame.

In [ ]:
figure = wh_plots.plot_footprint_diagnostics(
    footprint, stack, features, manifest, cfg
)
plt.show()

## What the temporal and harmonic features actually look like

Per-pixel feature maps. Features that vary by month are shown as their median across
months, noted in the panel title.

Read these as: the *model-free* features (`seasonal_range`, `wet_max`, `dry_median`) should
show the basin clearly; the *harmonic* features (`amplitude`, `phase`, `trend_per_year`)
are the ones under evaluation.

In [ ]:
MODEL_FREE = [
    "mndwi_seasonal_range", "mndwi_wet_max", "mndwi_dry_min",
    "ndvi_dry_median", "ndvi_seasonal_range", "ndti_seasonal_range",
    "mndwi_rank", "months_since_water",
]

figure = wh_plots.plot_feature_maps(features, MODEL_FREE, mask=footprint.mask)
plt.show()

In [ ]:
# The harmonic block is disabled by default (features.temporal.harmonic_enabled),
# because the ablation showed it hurting: waterholes fill sharply and drain
# slowly, which a sinusoid fits badly. When it is off these features do not
# exist, so there is nothing to plot.
HARMONIC = [
    "mndwi_harm_amplitude", "mndwi_harm_phase", "mndwi_harm_trend_per_year",
    "mndwi_harm_residual", "ndvi_harm_amplitude", "ndvi_harm_phase",
    "ndvi_harm_trend_per_year", "ndvi_harm_residual",
]

if cfg["features"]["temporal"]["harmonic_enabled"]:
    figure = wh_plots.plot_feature_maps(features, HARMONIC, mask=footprint.mask)
    plt.show()
else:
    print("harmonic_enabled is false, so no harmonic features were computed.")
    print("Set it true in waterhole_seg_config.yaml and re-run the site cell to see these.")
    print(f"\navailable feature blocks: "
          f"{sorted({n.split('_harm_')[0] for n in features if '_harm_' in n}) or 'none (harmonic off)'}")

## The harmonic fit on a single pixel

This is the plot to read before deciding what the harmonic block is worth.

Blue dots are wet-season observations, orange dots dry-season, the black line is the
fitted harmonic, and the red whiskers are the residuals. Dashed horizontal lines are that
pixel's own wet-season maximum and dry-season minimum.

A basin pixel fills sharply and drains slowly — an asymmetric sawtooth. The sinusoid cannot
follow that, and the residual panel shows exactly where it fails and by how much. **That
failure is informative**: `harm_residual` is the feature that carries sharp change, saying
"this pixel is far from its own seasonal norm right now".

Compare a basin pixel against a matrix pixel to see the difference.

In [ ]:
# The most basin-like pixel. If the footprint failed for this site, this falls
# back to the strongest-scoring pixel anywhere on the tile — which is exactly
# what you want to look at to understand WHY it failed.
basin_row, basin_col, note = wh_footprint.strongest_pixel(footprint)
print(note, f"-> ({basin_row}, {basin_col})")

figure = wh_plots.plot_pixel_timeseries(stack, features, basin_row, basin_col, cfg=cfg)
figure.suptitle(f"site {SITE} — BASIN pixel ({basin_row}, {basin_col})", fontsize=11)
plt.show()

In [ ]:
# A well-observed savanna-matrix pixel, for contrast.
observed_months = np.isfinite(stack.stacks["mndwi"]).sum(axis=0)
matrix_row, matrix_col, note = wh_footprint.background_pixel(footprint, observed_months)
print(note, f"-> ({matrix_row}, {matrix_col})")

figure = wh_plots.plot_pixel_timeseries(stack, features, matrix_row, matrix_col, cfg=cfg)
figure.suptitle(f"site {SITE} — MATRIX pixel ({matrix_row}, {matrix_col})", fontsize=11)
plt.show()

## How well does the score separate basin from matrix?

The basin/matrix ratio for each candidate layer, measured on this site. This is the number
that decides which layers deserve weight in `seasonal_range_weights`.

When the harmonic block is enabled, the scatter below also compares harmonic amplitude
against the model-free seasonal range — both claim to measure "how much does this pixel swing
seasonally", and whichever separates the basin further is the one worth keeping. Measured
earlier on site 025: seasonal range 2.85x, harmonic amplitude 1.73x, which is part of why the
harmonic is off by default.

In [ ]:
if not footprint.mask.any():
    print(f"site {SITE} has no footprint, so there is no basin/matrix split to compare.")
    print(f"reason: {footprint.reason}")
else:
    inside = footprint.mask
    harmonic_on = cfg["features"]["temporal"]["harmonic_enabled"]

    print(f"{'feature':32s} {'basin':>8s} {'matrix':>8s} {'ratio':>7s}")
    for index_name in ("mndwi", "ndvi", "ndti"):
        candidates = [f"{index_name}_seasonal_range", f"{index_name}_dry_median"]
        if harmonic_on:
            candidates.append(f"{index_name}_harm_amplitude")

        for feature in candidates:
            if feature not in features:
                continue
            outside = ~inside & np.isfinite(features[feature])
            basin = np.nanmedian(features[feature][inside])
            matrix = np.nanmedian(features[feature][outside])
            ratio = basin / matrix if matrix else np.nan
            print(f"{feature:32s} {basin:8.3f} {matrix:8.3f} {ratio:7.2f}x")

    if harmonic_on:
        figure = wh_plots.plot_harmonic_vs_model_free(
            features, index_name="mndwi", mask=footprint.mask
        )
        plt.show()
    else:
        print("\nharmonic_enabled is false, so there is no harmonic amplitude to compare.")

## Compare several sites

Run the chosen sites and look at the outlines side by side before committing to a full
run. This is where you tune `score_threshold` and `buffer_px`.

In [ ]:
for site_id in EXPLORE_SITES:
    try:
        site_footprint, site_stack, site_features = wh_footprint.run_site(
            manifest, site_id, cfg, PARAMS
        )
    except OSError as error:
        print(f"site {site_id}: SKIPPED — {error}")
        continue

    status = "OK" if site_footprint.succeeded else f"FAILED — {site_footprint.reason}"
    print(f"site {site_id}: {site_footprint.n_pixels:4d} px  "
          f"{site_footprint.area_m2/1e4:5.2f} ha  {status}")

    figure = wh_plots.plot_footprint_diagnostics(
        site_footprint, site_stack, site_features, manifest, cfg
    )
    plt.show()

## Run every site and save

Writes, per site, a uint8 GeoTIFF mask on that site's exact grid plus a WGS84 GeoJSON
polygon carrying the parameters that produced it, into `derived/footprints/`.

Sites that fail are reported rather than silently given an empty mask — an empty footprint
would become a zero denominator downstream, which is worse than a missing one.

Expect roughly 2 seconds per site when the tiles are hydrated locally. If chips are
OneDrive placeholders this will be far slower and may raise; see the note at the end.

In [ ]:
results = []
failures = []

for position, site_id in enumerate(sites, start=1):
    try:
        site_footprint, site_stack, site_features = wh_footprint.run_site(
            manifest, site_id, cfg, PARAMS
        )
    except OSError as error:
        failures.append((site_id, str(error).splitlines()[0]))
        continue

    if site_footprint.succeeded:
        wh_footprint.save_footprint(site_footprint, cfg)

    results.append({
        "site_id": site_id,
        "n_pixels": site_footprint.n_pixels,
        "area_ha": site_footprint.area_m2 / 1e4,
        "fraction_of_tile": site_footprint.fraction_of_tile,
        "succeeded": site_footprint.succeeded,
        "reason": site_footprint.reason,
        "notes": site_footprint.notes,
    })

    if position % 20 == 0:
        print(f"  {position}/{len(sites)} sites")

results = pd.DataFrame(results)
results.to_csv(cfg.paths["derived"] / "footprints_summary.csv", index=False)

print(f"\n{results['succeeded'].sum()} of {len(results)} sites footprinted")
if failures:
    print(f"{len(failures)} site(s) could not be read:")
    for site_id, message in failures[:10]:
        print(f"  {site_id}: {message}")

## Summary

Footprint size distribution and the reasons sites failed. A cluster of very small
footprints is worth a look — those are sites where the basin may be too small at 10 m to
be worth labelling at all.

In [ ]:
figure = wh_plots.plot_footprint_summary(results)
plt.show()

print(results[results["succeeded"]]["n_pixels"].describe().round(1).to_string())
print()
print("failures by reason:")
print(results[~results["succeeded"]]["reason"].value_counts().to_string())

In [ ]:
# Sites with a usable basin, largest first — the sensible order to label in.
labellable = results[results["succeeded"]].sort_values("n_pixels", ascending=False)
labellable.head(20)

---

### If chips fail to open

The tiles live on a OneDrive share with Files On-Demand, which dehydrates files to
placeholders. A dehydrated chip reports a normal file size but cannot be opened, surfacing
as `errno 60` or *"not recognized as a supported file format"*. Retrying does not help.

Fix: right-click `cookie-cutting` in Finder and choose **Always Keep on This Device**, or
move `images_tif_v2` off OneDrive entirely. `wh_tiles.read_tile` detects this case and says
so explicitly rather than raising the misleading underlying error.